In [ ]:
#!/usr/bin/env python
# coding: utf-8

import os
import torch
from torch.utils.data import Dataset
import pydicom
import numpy as np
import torch.nn.functional as F
from skimage.util import img_as_ubyte
from skimage.filters.rank import entropy
from skimage.morphology import disk
import cv2

class CT2DSliceDataset(Dataset):
    """
    Returns:
      ldct_padded: (1, Hp, Wp) float32 in [0,1]
      noise_padded: (1, Hp, Wp) float32 in [0,1]
      ndct_padded: (1, Hp, Wp) float32 in [0,1]
      region_padded: (R, Hp, Wp) float32 where each channel r = ldct * binary_mask_r
                     (i.e. intensity maps per region)
    """
    def __init__(self, root_dir, patch_pad=32):
        self.root_dir = root_dir
        self.patch_pad = patch_pad
        self.image_pairs = self._collect_dicom_pairs()
        print(f"[INFO] Found {len(self.image_pairs)} matched LDCT-NDCT pairs.")

    def _collect_dicom_pairs(self):
        pairs = []
        for patient_id in os.listdir(self.root_dir):
            patient_path = os.path.join(self.root_dir, patient_id)
            if not os.path.isdir(patient_path):
                continue

            ldct_dir = os.path.join(patient_path, "quarter_1mm")
            ndct_dir = os.path.join(patient_path, "full_1mm")

            if not (os.path.isdir(ldct_dir) and os.path.isdir(ndct_dir)):
                print(f"[WARN] Skipping {patient_id}: Missing folders.")
                continue

            ldct_files = [
                os.path.join(ldct_dir, f)
                for f in os.listdir(ldct_dir)
                if f.endswith(".IMA")
            ]
            ndct_files = [
                os.path.join(ndct_dir, f)
                for f in os.listdir(ndct_dir)
                if f.endswith(".IMA")
            ]

            def sort_by_instance(files):
                def instnum(path):
                    try:
                        return int(pydicom.dcmread(path).InstanceNumber)
                    except Exception:
                        return os.path.basename(path)
                return sorted(files, key=instnum)

            ldct_sorted = sort_by_instance(ldct_files)
            ndct_sorted = sort_by_instance(ndct_files)

            if len(ldct_sorted) != len(ndct_sorted):
                print(f"[WARN] Mismatch in slice count for {patient_id}")
                continue

            pairs.extend(list(zip(ldct_sorted, ndct_sorted)))
        return pairs

    def _load_dicom(self, path):
        dcm = pydicom.dcmread(path)
        img = dcm.pixel_array.astype(np.float32)

        slope = getattr(dcm, 'RescaleSlope', 1.0)
        intercept = getattr(dcm, 'RescaleIntercept', 0.0)
        hu = img * slope + intercept  # Hounsfield Units

        clipped = np.clip(hu, -1024, 3072)
        minv = clipped.min()
        maxv = clipped.max()
        if maxv - minv < 1e-6:
            # constant image -> produce zeros
            norm = np.zeros_like(clipped, dtype=np.float32)
        else:
            norm = (clipped - minv) / (maxv - minv)

        return hu.astype(np.float32), norm.astype(np.float32)

    def _apply_padding(self, img_tensor):
        return F.pad(img_tensor, pad=[self.patch_pad]*4, mode='constant', value=0)

    def compute_noise_map(self, image_np):
        # image_np is expected in [0,1]
        try:
            img = img_as_ubyte(np.clip(image_np, 0.0, 1.0))
            entropy_map = entropy(img, disk(5))
            norm_entropy = (entropy_map - entropy_map.min()) / (entropy_map.max() - entropy_map.min() + 1e-8)
            return norm_entropy.astype(np.float32)
        except Exception:
            # fallback to zeros
            h, w = image_np.shape[:2]
            return np.zeros((h, w), dtype=np.float32)

    def compute_binary_region_masks(self, hu_image):
        # produce binary masks same as before
        lung_mask = (hu_image >= -1000) & (hu_image <= -500)
        soft_tissue_mask = (hu_image > -150) & (hu_image < 150)
        bone_mask = (hu_image >= 300)

        def refine(mask):
            kernel = np.ones((5, 5), np.uint8)
            refined = cv2.morphologyEx(mask.astype(np.uint8), cv2.MORPH_CLOSE, kernel)
            return refined.astype(np.float32)

        lung_ref = refine(lung_mask)
        soft_ref = refine(soft_tissue_mask)
        bone_ref = refine(bone_mask)

        exclus_lung = lung_ref.copy()
        exclus_bone = bone_ref.copy()
        exclus_soft = soft_ref.copy()

        exclus_bone[exclus_lung == 1] = 0
        exclus_soft[exclus_lung == 1] = 0
        exclus_soft[exclus_bone == 1] = 0

        masks = np.stack([exclus_lung, exclus_soft, exclus_bone], axis=0)
        return masks.astype(np.float32)

    def compute_region_intensity_maps(self, image_norm, binary_masks):
        """
        image_norm: (H, W) normalized image in [0,1]
        binary_masks: (R, H, W) binary masks 0/1
        returns region_maps (R, H, W) where each channel = image_norm * mask
        """
        # ensure shapes
        R = binary_masks.shape[0]
        maps = np.zeros_like(binary_masks, dtype=np.float32)
        for r in range(R):
            maps[r] = image_norm * binary_masks[r]
        return maps.astype(np.float32)

    def __len__(self):
        return len(self.image_pairs)

    def __getitem__(self, idx):
        ldct_path, ndct_path = self.image_pairs[idx]

        ldct_hu, ldct_img = self._load_dicom(ldct_path)
        ndct_hu, ndct_img = self._load_dicom(ndct_path)

        noise_map = self.compute_noise_map(ldct_img)
        binary_masks = self.compute_binary_region_masks(ldct_hu)
        region_maps = self.compute_region_intensity_maps(ldct_img, binary_masks)

        ldct_tensor = torch.tensor(ldct_img, dtype=torch.float32).unsqueeze(0)   # (1,H,W)
        ndct_tensor = torch.tensor(ndct_img, dtype=torch.float32).unsqueeze(0)
        noise_tensor = torch.tensor(noise_map, dtype=torch.float32).unsqueeze(0)
        region_tensor = torch.tensor(region_maps, dtype=torch.float32)           # (R,H,W)

        ldct_padded = self._apply_padding(ldct_tensor)
        ndct_padded = self._apply_padding(ndct_tensor)
        noise_padded = self._apply_padding(noise_tensor)
        region_padded = F.pad(region_tensor, pad=[self.patch_pad]*4, mode='constant', value=0)

        return ldct_padded, noise_padded, ndct_padded, region_padded
